# BYVA Phase 3 — Colab T4 training run

Runs the real Phase 3 gate on a Colab NVIDIA T4 (16 GB): BPE 32k tokenizer
training, then a SMALL-config pretraining run (D-0013) with FP16 and gradient
accumulation (D-0018) on Phase 2 shards.

Before starting, read `docs/PHASES.md` (gate definition), `docs/PHASE3_REPORT.md`
(status), and `docs/DECISIONS.md` (D-0013..D-0018). Record only results that
actually happen in this session.

In [1]:
REPO_BRANCH = "vibe/bpe-trainer-colab-54b751"
!rm -rf /content/byva-v1
!git clone --depth 1 -b {REPO_BRANCH} https://github.com/val-alpha/byva-v1.git /content/byva-v1
%cd /content/byva-v1
!git log --oneline -1
import pathlib
missing = [p for p in ("training/train_bpe.py", "training/byva_tokenizer.py") if not pathlib.Path(p).is_file()]
train_cli = pathlib.Path("training/train.py").read_text(encoding="utf-8")
if missing or "--tokenizer" not in train_cli:
    raise RuntimeError(
        f"clone of {REPO_BRANCH} lacks the Colab trainer code "
        f"(missing: {missing or 'train.py --tokenizer'}); fix REPO_BRANCH"
    )
print("repo OK for the Colab T4 run")

Cloning into '/content/byva-v1'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 83 (delta 4), reused 31 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 90.62 KiB | 786.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/byva-v1
968312f (grafted, HEAD -> vibe/bpe-trainer-colab-54b751, origin/vibe/bpe-trainer-colab-54b751) data: stream progress output from build_shards; colab: guard prerequisite cells
repo OK for the Colab T4 run


In [2]:
import torch, subprocess
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
!nvidia-smi

2.14.1+cu130 True
Sat Nov 28 05:15:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------

## 1. Data — download admitted sources (docs/DATA.md)

All four sources below are admitted and ungated. Each cell streams/samples a
slice and writes plain JSONL `{"text": ...}`; `build_shards.py` then makes
deterministic shards. Set per-source document budgets so the recorded id:en
mixture (D-0016) matches your run manifest. HF token goes in Colab secrets as
`HF_TOKEN` (Secrets tab, left sidebar) — needed for high-rate downloads, not
for the ungated reads.

In [3]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
DOCS = {"wiki_id": 200_000, "wiki_en": 200_000, "c4_id": 200_000,
        "fw_edu_en": 200_000, "fw2_id": 200_000}
MIXTURE_NOTE = "id:en = " + str(sum(DOCS[k] for k in DOCS if k.endswith("_id") or k == "fw2_id")) + ":" + str(sum(DOCS[k] for k in DOCS if k.endswith("_en")))
print(MIXTURE_NOTE)

id:en = 600000:400000


In [4]:
# 1a. Wikipedia id + en (CC BY-SA 3.0, wikimedia/wikipedia)
from datasets import load_dataset
import json
import os

# Memastikan direktori tujuan ada
os.makedirs("/content/data", exist_ok=True)

for cfg, name in (("20231101.id", "wiki_id"), ("20231101.en", "wiki_en")):
    ds = load_dataset("wikimedia/wikipedia", cfg, split="train", streaming=True)
    with open(f"/content/data/{name}.jsonl", "w", encoding="utf-8") as f:
        for i, row in enumerate(ds):
            if i >= DOCS[name]: break
            f.write(json.dumps({"text": row["text"]}, ensure_ascii=False) + "\n")
    print(name, "done")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

wiki_id done


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

wiki_en done


In [5]:
# 1b. c4 id (ODC-BY 1.0, allenai/c4)
from datasets import load_dataset
import json
ds = load_dataset("allenai/c4", "id", split="train", streaming=True)
with open("/content/data/c4_id.jsonl", "w", encoding="utf-8") as f:
    for i, row in enumerate(ds):
        if i >= DOCS["c4_id"]: break
        f.write(json.dumps({"text": row["text"]}, ensure_ascii=False) + "\n")
print("c4_id done")

README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

c4_id done


In [6]:
# 1c. FineWeb-Edu en sample (ODC-BY 1.0) — one CC snapshot
from datasets import load_dataset
import json
ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)
with open("/content/data/fw_edu_en.jsonl", "w", encoding="utf-8") as f:
    for i, row in enumerate(ds):
        if i >= DOCS["fw_edu_en"]: break
        f.write(json.dumps({"text": row["text"]}, ensure_ascii=False) + "\n")
print("fw_edu_en done")

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

fw_edu_en done


In [7]:
# 1d. FineWeb-2 ind_Latn sample (ODC-BY 1.0) — the ungated Indonesian source
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq, json
files = ["000_00000.parquet", "000_00001.parquet"]
written = 0
with open("/content/data/fw2_id.jsonl", "w", encoding="utf-8") as out:
    for name in files:
        if written >= DOCS["fw2_id"]: break
        local = hf_hub_download("HuggingFaceFW/fineweb-2",
                                f"data/ind_Latn/train/{name}", repo_type="dataset")
        pf = pq.ParquetFile(local)
        for batch in pf.iter_batches(batch_size=1_000, columns=["text"]):
            for text in batch.column("text").to_pylist():
                if written >= DOCS["fw2_id"]: break
                out.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
                written += 1
        del local
print("fw2_id done:", written, "docs")

data/ind_Latn/train/000_00000.parquet: reconstructing file:   0%|          |  0.00B / 4.85GB            

data/ind_Latn/train/000_00000.parquet: downloading bytes:           |  0.00B            

fw2_id done: 200000 docs


### 1e. OSCAR-2301 (optional — skip unless the manual gate approved your account)

CC0-1.0, bulk `.jsonl.zst` parts under `id_meta/` / `en_meta/`. The cell below
no-ops gracefully while access is pending.

In [8]:
# 1e. OSCAR-2301 (optional)
import urllib.request, os, json
token = os.environ.get("HF_TOKEN", "")
probe = urllib.request.Request(
    "https://huggingface.co/datasets/oscar-corpus/OSCAR-2301/resolve/main/id_meta/id_meta_part_1.jsonl.zst")
probe.add_header("Authorization", f"Bearer {token}")
probe.add_header("Range", "bytes=0-63")
try:
    urllib.request.urlopen(probe, timeout=30)
    print("OSCAR access OK — add a download cell per docs/DATA.md layout (id_meta/*.jsonl.zst)")
except Exception as e:
    print("OSCAR not accessible yet (pending gate approval):", type(e).__name__)
    # do not fail the run; OSCAR is optional for this gate
    pass

OSCAR not accessible yet (pending gate approval): HTTPError


## 1f. Build deterministic shards (D-0012)

In [9]:
SHARD_DIR = "/content/shards"
import pathlib
if (pathlib.Path(SHARD_DIR) / "manifest.json").is_file():
    print(f"shards already present in {SHARD_DIR}; skipping rebuild")
else:
    print("building shards from ~1M docs; several minutes total, DO NOT press stop (progress + elapsed time prints continuously)")
    print("if it ever gets interrupted, just run this cell again: finished shards are kept and the run resumes")
    !python3 training/build_shards.py \
      --input /content/data/wiki_id.jsonl --input /content/data/c4_id.jsonl --input /content/data/fw2_id.jsonl \
      --input /content/data/wiki_en.jsonl --input /content/data/fw_edu_en.jsonl \
      --output-dir {SHARD_DIR}
!cat {SHARD_DIR}/manifest.json | head -40


building shards from ~1M docs; this can take 10-30+ min, WAIT for it to finish (progress prints every 50k docs)
loading wiki_id.jsonl ...
  wiki_id.jsonl: 50000 docs loaded
  wiki_id.jsonl: 100000 docs loaded
  wiki_id.jsonl: 150000 docs loaded
  wiki_id.jsonl: 200000 docs loaded
loaded 200000 docs from wiki_id.jsonl
loading c4_id.jsonl ...
  c4_id.jsonl: 50000 docs loaded
  c4_id.jsonl: 100000 docs loaded
  c4_id.jsonl: 150000 docs loaded
  c4_id.jsonl: 200000 docs loaded
loaded 200000 docs from c4_id.jsonl
loading fw2_id.jsonl ...
  fw2_id.jsonl: 50000 docs loaded
  fw2_id.jsonl: 100000 docs loaded
  fw2_id.jsonl: 150000 docs loaded
  fw2_id.jsonl: 200000 docs loaded
loaded 200000 docs from fw2_id.jsonl
loading wiki_en.jsonl ...
  wiki_en.jsonl: 50000 docs loaded
  wiki_en.jsonl: 100000 docs loaded
  wiki_en.jsonl: 150000 docs loaded
  wiki_en.jsonl: 200000 docs loaded
loaded 200000 docs from wiki_en.jsonl
loading fw_edu_en.jsonl ...
  fw_edu_en.jsonl: 50000 docs loaded
  fw_edu_en.j

## 2. Train the 32,768 BPE table (prerequisite, D-0015)

In [10]:
ARTIFACT = "/content/byva-tokenizer-32k-v1.json"
import pathlib
if not (pathlib.Path(SHARD_DIR) / "manifest.json").is_file():
    raise RuntimeError(
        f"no manifest.json in {SHARD_DIR}; run the build-shards cell first and let it finish "
        "(it prints progress every 50k docs and can take 10-30+ min on ~1M docs)"
    )
!python3 training/train_bpe.py --shards {SHARD_DIR} --target-vocab 32768 --version 1 --output {ARTIFACT} --stats /content/bpe-stats.json
!cat /content/bpe-stats.json


RuntimeError: no manifest.json in /content/shards; run the build-shards cell first and let it finish (it prints progress every 50k docs and can take 10-30+ min on ~1M docs)

## 3. SMALL training run on the T4 (FP16 + gradient accumulation)

In [ ]:
CKPT = "/content/drive/MyDrive/byva/ckpt-small-t4.pt"
import pathlib
if not pathlib.Path(ARTIFACT).is_file():
    raise RuntimeError(f"tokenizer artifact {ARTIFACT} missing; run the BPE training cell first")
from google.colab import drive
drive.mount('/content/drive')
!python3 training/train.py --shards {SHARD_DIR} --config small --tokenizer {ARTIFACT} --device cuda --fp16 --grad-accum 4 --steps 10000 --seq-len 1024 --batch-size 8 --checkpoint {CKPT}


### Resume after a disconnected session (D-0014)

In [ ]:
!python3 training/train.py --shards {SHARD_DIR} --config small --tokenizer {ARTIFACT} --device cuda --fp16 --grad-accum 4 --steps 10000 --seq-len 1024 --batch-size 8 --checkpoint {CKPT} --resume

## 4. Record the gate evidence

In [ ]:
import torch, dataclasses, sys
sys.path.insert(0, "training")
from train import load_checkpoint
model, optimizer, state, scaler = load_checkpoint(CKPT)
print("steps:", state.step)
print("loss:", state.loss_history[0], "->", state.loss_history[-1])
print("config:", dataclasses.asdict(model.config))
print("mixture:", MIXTURE_NOTE)

Copy the printed evidence into `docs/PHASE3_REPORT.md` (loss curve, val loss,
generation sample) and the agent progress note. The Phase 3 gate passes only if
loss actually fell on held-out data with the trained BPE tokenizer.